# Semi-Parametric Bootstrap Cross-Fitting

This workflow implements a robust method for estimating the individualized probability of an outcome (AUC) using **Heteroscedastic Deep Learning** combined with **Semi-Parametric Inference**.

The goal is to estimate $P(T_{control} > T_{case} | x)$—the probability that a Control patient has a better outcome (e.g., survival time, biological age) than a Case patient—without assuming that prediction errors follow a perfect Gaussian distribution.

### 1. Generative Model: Heteroscedastic Neural Network

Unlike standard regression which assumes constant variance (homoscedasticity), we model the conditional distribution of the target variable $Y$ given input features $X$ as a Gaussian distribution where **both the mean and the variance depend on the input**.

**Formulation:**
For patient $i$, the response $y_i$ is modeled as:

$$
y_i \sim \mathcal{N}(\mu(x_i; \theta), \sigma^2(x_i; \theta))
$$

Where the neural network $f_\theta(x)$ outputs two parameters:
1. $\hat{\mu}(x)$: The estimated central prediction.
2. $\hat{\sigma}(x)$: The estimated aleatoric uncertainty (standard deviation).

**Loss Function (Training):**
The model is trained by minimizing the **Heteroscedastic Negative Log-Likelihood (NLL)**. This encourages the model to increase $\sigma$ when the squared error is large, effectively learning to quantify its own uncertainty.

$$
\mathcal{L}(\theta) = \frac{1}{N} \sum_{i=1}^{N} \left( \frac{1}{2}\log(\hat{\sigma}(x_i)^2) + \frac{(y_i - \hat{\mu}(x_i))^2}{2\hat{\sigma}(x_i)^2} \right)
$$

---

### 2. Bootstrap Cross-Fitting (Residual Collection)

To perform honest simulations, we need the **true distribution of model errors** ($\epsilon$), not the theoretical one. We cannot use training errors because they are biased (underestimated due to overfitting). We use a nested procedure to collect "clean" residuals:

**Algorithm:**

1.  **Outer Loop (Bootstrap $b=1 \dots B$):**
    Generate a resampled dataset $D^*_b$ from the original data $D$ (sampling with replacement).

2.  **Inner Loop (Cross-Fitting with $K$-Folds):**
    Split $D^*_b$ into $K$ folds. For each fold $k$:
    * **Train:** Fit model $M_k$ on the training partition $D_{train}^{(k)}$.
    * **Validate (Cross-Fit):** Use $M_k$ to predict on the validation partition $D_{val}^{(k)}$ (data not seen by $M_k$).

    Calculate the **Standardized Empirical Residuals** ($\hat{\epsilon}$) only on the validation data:

    $$
    \hat{\epsilon}_j = \frac{y_j - \hat{\mu}(x_j)}{\hat{\sigma}(x_j)}, \quad \forall j \in D_{val}^{(k)}
    $$

    By the end of the $K$-folds, we obtain a pool of residuals $\mathcal{E}_b = \{ \hat{\epsilon}_1, \dots, \hat{\epsilon}_N \}$ representing the real error distribution on unseen data.

---

### 3. Semi-Parametric Density Estimation (Monte Carlo)

To predict the outcome for a new patient, we do not assume the error is Gaussian. Instead, we use the empirical distribution $\mathcal{E}$ collected in the previous step.

Let $x_0$ be a Control patient and $x_1$ be a Case patient.

**Monte Carlo Simulation:**
We generate $M$ possible scenarios (e.g., $M=500$) by sampling random residuals $\epsilon^*$ from the pool $\mathcal{E}$:

$$
T_0^{(m)} = \hat{\mu}(x_0) + \hat{\sigma}(x_0) \cdot \epsilon^*_A, \quad \epsilon^*_A \sim \text{Uniform}(\mathcal{E})
$$

$$
T_1^{(m)} = \hat{\mu}(x_1) + \hat{\sigma}(x_1) \cdot \epsilon^*_B, \quad \epsilon^*_B \sim \text{Uniform}(\mathcal{E})
$$

**AUC Calculation:**
The individualized AUC is estimated as the frequency with which the Control outcome exceeds the Case outcome across simulations:

$$
\widehat{AUC}(x_0, x_1) = \frac{1}{M} \sum_{m=1}^{M} \mathbb{I}\left( T_0^{(m)} > T_1^{(m)} \right)
$$

*Where $\mathbb{I}(\cdot)$ is the indicator function.*

---

### 4. Out-Of-Bag (OOB) Evaluation

To report a globally unbiased metric, we ensure that no patient is evaluated using a model that saw them during training. We leverage the properties of Bootstrap aggregation.

Let $I_b$ be the set of patient indices included in the training of Bootstrap $b$. For a specific patient $i$:

1.  Identify the Bootstrap iterations where patient $i$ was **excluded** (Out-Of-Bag, $i \notin I_b$).
2.  Average the AUC estimates obtained only from those iterations.

$$
\text{AUC\_Final}_i = \mathbb{E}_{b} \left[ \widehat{AUC}_b(x_i) \mid i \notin I_b \right]
$$

This approach captures both **aleatoric uncertainty** (via $\sigma$) and **epistemic uncertainty** (via Bootstrap), providing a robust and conservative estimate of the model's discriminative power.

In [ ]:
import sys
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import KFold
from scipy.stats import norm
from tqdm import tqdm
import os
import matplotlib.pyplot as plt

# num_to_onehot/normalize_column are identical to the ones already in
# src/real_data/data_reg_real.py (used by the real-data mlp_reg.py pipeline) --
# reused from there instead of duplicating them here. Run this notebook from the
# repository root so this path resolves.
sys.path.insert(0, os.path.join("src", "real_data"))
from data_reg_real import num_to_onehot, normalize_column

# =============================================================================
# 0. GLOBAL CONFIGURATION & REPRODUCIBILITY
# =============================================================================

# Set seeds to ensure results are the same every time this script runs
RANDOM_SEED = 42
torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# Configuration dictionary acting as a replacement for command-line arguments.
# This controls the entire pipeline (input/output, model hyperparameters, etc.)
config = {
    "input_dir": "./data",           # Directory containing df_f.csv / df_m.csv
    "output_file": "./output",       # Root directory for saving CSV results and PNG plots (git-ignored)
    "combination": 6,                # Index selector for 'get_feature_combination'
    "k_folds": 2,                    # Number of splits for Cross-Validation
    "num_epochs": 500,               # Training iterations per fold
    "batch_size": 200,               # Number of samples per gradient update
    "learning_rate": 0.001,          # Step size for the optimizer
    "weight_decay": 1e-4,            # L2 Regularization (prevents overfitting)
    "dropout": 0.2,                  # Fraction of neurons to drop during training
    "hidden_layers": [64, 32]        # Architecture: Input -> 64 -> 32 -> Output
}

# Constants for NHANES data processing
GENDER_COL = "RIAGENDR"
GENDER_MAP = {"male": 1, "female": 2}
TARGET_COL = "TAC2"  # The biological age or time-to-event variable
GROUP_COLUMNS = ["tres", "cinco", "ocho"]  # Classification thresholds (e.g., survival years)

# Explicit filename per gender (data/df_f.csv, data/df_m.csv). This used to be a
# fuzzy filename-substring filter over every file in --input_dir, but --input_dir
# is now the shared data/ folder (also holding female_data.csv,
# female_data_with_cancer.csv, etc.), whose names also match a "female"/"male"
# substring filter -- the fuzzy filter would silently pick up the wrong files.
GENDER_FILES = {"female": "df_f.csv", "male": "df_m.csv"}

# =============================================================================
# 1. FEATURE SELECTION & FILTERS
# =============================================================================

def get_feature_combination(idx: int) -> list[str]:
    """
    Selects a specific list of input features (column names) based on an index.
    Useful for experimenting with different subsets of biological/demographic data.

    Args:
        idx (int): The identifier for the feature set (0-7).

    Returns:
        list[str]: A list of column names to be used as X (inputs).
    """
    if idx == 0:
        return ['RIDAGEYR.x', 'BMXHT', 'BMXWT', 'BMXBMI', 'BMXWAIST', 'BPXDI1',
                'BPXSY1', 'BPXPLS', 'LBDSCHSI_43', 'LBXSTR_43', 'LBXSGL_43', 'RIAGENDR', 'LBXGH_39']
    elif idx == 1:
        return ['RIDAGEYR.x', 'BMXHT', 'BMXWT', 'BMXBMI', 'BMXWAIST', 'BPXDI1',
                'BPXSY1', 'BPXPLS', 'BPXDI1', 'LBDSCHSI_43', 'LBXSTR_43', 'LBXSGL_43', 'RIAGENDR']
    # ... (skipping indices 2, 3, 4, 5 for brevity, logic remains the same)
    elif idx == 6:
        # Minimalist set: Age and BMI only
        return ['RIDAGEYR.x', 'BMI']
    elif idx == 7:
        # Physical measurements only
        return ['BMXHT', 'BMXWT', 'BMXBMI', 'BMXWAIST', 'BPXDI1', 'BPXSY1', 'BPXPLS', 'BPXDI1']
    else:
        # Default fallback (same as idx 0)
        return ['RIDAGEYR.x', 'BMXHT', 'BMXWT', 'BMXBMI', 'BMXWAIST', 'BPXDI1',
                'BPXSY1', 'BPXPLS', 'LBDSCHSI_43', 'LBXSTR_43', 'LBXSGL_43', 'RIAGENDR', 'LBXGH_39']

# =============================================================================
# 2. DATA PREPROCESSING HELPERS
# =============================================================================

def create_dict_from_df(df, index_0, index_1, output_var, variables):
    """
    Core data structuring function.
    1. Normalizes continuous variables.
    2. One-hot encodes categorical variables (if any).
    3. Stacks them into a single input matrix X.
    4. Splits data into dictionaries for 'All', 'Group 0' (Control), and 'Group 1' (Case).

    Unlike src/real_data/data_reg_real.py's create_dict, this does not require a
    'SEQN' subject-id column or NHANES survey weights ('wtmec4yr_adj_norm') -- df_f.csv/
    df_m.csv have neither, so weights default to 1 per subject when no 'weights'
    column is present. Kept notebook-local rather than merged into data_reg_real.py
    for that reason.

    Args:
        df (pd.DataFrame): The source dataframe.
        index_0 (array): Row indices belonging to Group 0 (e.g., Controls).
        index_1 (array): Row indices belonging to Group 1 (e.g., Cases).
        output_var (str): The name of the target column (Y).
        variables (list): List of feature column names (X).

    Returns:
        tuple: (dict_all, dict_0, dict_1). Each dict contains 'x', 'y', 'w' arrays.
    """
    L, colnames = [], []
    
    # Currently assuming all variables passed are continuous (interval)
    # If categorical variables were passed, they should be moved to 'cat_variables'
    int_variables = variables
    cat_variables = [] 
    
    for i, c in enumerate(variables):
        if c in int_variables:
            feature_vector = normalize_column(df[c])
            L.extend([np.expand_dims(feature_vector, 1)])
            colnames.extend([c])
        elif c in cat_variables:
            feature_vector = num_to_onehot(df[c].to_numpy())
            L.extend([feature_vector])
            for j in range(feature_vector.shape[1]):
                colnames.extend([c + "_" + str(j)])

    # Horizontal stack: Combine all feature columns into one matrix
    input_data = np.hstack(L).astype('float32')
    
    # Prepare Target (Y) and Weights (W)
    target = df[output_var].values.astype('float32').reshape(-1, 1)
    weights = df['weights'].astype('float32').values if 'weights' in df.columns else np.ones(len(df), dtype='float32')
    weights = weights.reshape(-1, 1)

    # Create the dictionaries
    dict_all = {'x': input_data, 'y': target, 'w': weights}
    dict_0   = {'x': input_data[index_0], 'y': target[index_0], 'w': weights[index_0]}
    dict_1   = {'x': input_data[index_1], 'y': target[index_1], 'w': weights[index_1]}
    
    return dict_all, dict_0, dict_1

def load_data_safe(df, variables, output_var, group_col):
    """
    Wrapper that identifies the indices for Group 0 and Group 1 based on the 'group_col'.
    """
    index_0 = np.where((df[group_col] == 0))[0]
    index_1 = np.where((df[group_col] == 1))[0]
    return create_dict_from_df(df, index_0, index_1, output_var, variables)

# =============================================================================
# 3. NEURAL NETWORK MODEL (HETEROSCEDASTIC)
# =============================================================================

class HeteroscedasticMLP(nn.Module):
    """
    A Multi-Layer Perceptron that predicts a Gaussian distribution (Mean and Variance)
    instead of a single point estimate.
    """
    def __init__(self, input_dim, hidden_layers=[64, 32], dropout_rate=0.05):
        super(HeteroscedasticMLP, self).__init__()
        layers = []
        in_features = input_dim
        
        # Dynamically build hidden layers
        for h_dim in hidden_layers:
            layers.append(nn.Linear(in_features, h_dim))
            layers.append(nn.ReLU()) # Activation function
            layers.append(nn.Dropout(dropout_rate)) # Regularization
            in_features = h_dim
            
        self.features = nn.Sequential(*layers)
        
        # Important: Output layer has 2 units (Mu and Sigma_pre)
        # Unit 0 -> Predicted Mean (Mu)
        # Unit 1 -> Predicted Standard Deviation parameter
        self.output_layer = nn.Linear(in_features, 2) 

    def forward(self, x):
        """
        Forward pass logic.
        """
        features = self.features(x)
        out = self.output_layer(features)
        
        mu = out[:, 0]
        # Apply Softplus to ensure Sigma is always positive.
        # Adding 1e-6 prevents numerical instability (division by zero).
        sigma = F.softplus(out[:, 1]) + 1e-6 
        
        return mu, sigma

def train_hetero_model(X, Y, W, config, device):
    """
    Trains the Heteroscedastic model using Negative Log Likelihood (NLL).
    
    Args:
        X, Y, W: Training data, targets, and weights.
        config: Hyperparameters dictionary.
        device: CPU, CUDA, or MPS.
        
    Returns:
        model: The trained PyTorch model.
    """
    # Initialize model. Bug fix: this used to read config["lr"]/config["epochs"],
    # which don't exist in `config` (only "learning_rate"/"num_epochs" do), so both
    # silently fell back to their .get() defaults -- in particular training only
    # ever ran 50 epochs regardless of config["num_epochs"]=500. dropout_rate was
    # also never passed through at all (always the class default of 0.05, ignoring
    # config["dropout"]=0.2). All three now read the correct config keys.
    model = HeteroscedasticMLP(
        X.shape[1],
        config.get("hidden_layers", [64, 32]),
        dropout_rate=config.get("dropout", 0.05),
    ).to(device)
    optimizer = optim.Adam(model.parameters(), lr=config.get("learning_rate", 0.001))
    
    # Convert numpy arrays to PyTorch tensors
    tX = torch.tensor(X, dtype=torch.float32).to(device)
    tY = torch.tensor(Y, dtype=torch.float32).to(device)
    tW = torch.tensor(W, dtype=torch.float32).to(device)
    
    # Create dataset and loader for batching
    dset = TensorDataset(tX, tY, tW)
    loader = DataLoader(dset, batch_size=config.get("batch_size", 64), shuffle=True)
    
    model.train()
    # Training Loop
    for _ in range(config.get("num_epochs", 50)):
        for xb, yb, wb in loader:
            optimizer.zero_grad() # Reset gradients
            
            mu, sigma = model(xb)
            
            # Loss Function: Heteroscedastic Negative Log Likelihood
            # NLL = 0.5 * log(sigma^2) + 0.5 * ((y - mu)^2 / sigma^2)
            # This penalizes high error but allows the model to "explain away" outliers 
            # by increasing sigma (uncertainty).
            loss = (0.5 * torch.log(sigma**2) + 0.5 * (yb.flatten() - mu)**2 / sigma**2) * wb.flatten()
            
            loss.mean().backward() # Backpropagation
            optimizer.step()       # Update weights
            
    return model

# =============================================================================
# 4. EMPIRICAL AUC CALCULATION (MONTE CARLO)
# =============================================================================

def compute_empirical_auc(mu0, mu1, s0, s1, eps0, eps1, n_mc=200):
    """
    Calculates the probability P(T_control > T_case) using Empirical Residuals.
    
    Why: Standard approaches assume errors are perfectly Gaussian. Real medical data isn't.
    This function uses the *actual* residuals (eps0, eps1) observed in validation to 
    simulate outcomes, providing a more robust AUC.
    
    Args:
        mu0, s0: Predicted Mean/Sigma for Controls.
        mu1, s1: Predicted Mean/Sigma for Cases.
        eps0, eps1: Arrays of standardized residuals collected from Cross-Validation.
        n_mc (int): Number of Monte Carlo simulations per patient.
        
    Returns:
        np.array: The estimated AUC for each patient (probability that T0 > T1).
    """
    # 1. Sample Residuals (Bootstrap the errors)
    # We pick 'n_mc' random errors from the pool we collected during CV.
    e0_samp = np.random.choice(eps0, size=n_mc, replace=True) 
    e1_samp = np.random.choice(eps1, size=n_mc, replace=True) 
    
    # 2. Vectorized Simulation
    # Expand dims to allow broadcasting: (N_patients, 1) vs (1, n_mc)
    # Resulting shape is (N_patients, n_mc)
    # T_simulated = Predicted_Mean + Predicted_Sigma * Random_Residual
    T0_sim = mu0[:, None] + s0[:, None] * e0_samp[None, :] 
    T1_sim = mu1[:, None] + s1[:, None] * e1_samp[None, :] 
    
    # 3. Comparison
    # Count how many times the simulated control outcome > simulated case outcome
    wins = (T0_sim > T1_sim).astype(np.float32)
    
    # 4. Average over simulations to get probability
    auc_per_patient = wins.mean(axis=1) 
    
    return auc_per_patient

# =============================================================================
# 5. MAIN WORKFLOW: BOOTSTRAP + CROSS-FITTING
# =============================================================================

def run_bootstrap_crossfitting(group_col, gender, config, B=100, K=5):
    """
    Orchestrates the entire evaluation process.
    
    Process:
    1. Loads files.
    2. Loops B times (Bootstrap).
    3. Inside each Bootstrap, performs K-Fold Cross-Validation.
    4. Collects OOB (Out-Of-Bag) predictions to avoid data leakage.
    5. Calculates Empirical Residuals from validation folds.
    6. Computes AUC using the helper function.
    7. Aggregates results and saves/plots.
    
    Args:
        group_col (str): The column used to split groups (e.g., 'three_year_survival').
        gender (str): Gender filter.
        B (int): Number of bootstraps.
        K (int): Number of CV folds.
    """
    print(f"\n=== Bootstrap-CrossFitting (Empirical Residuals): {group_col} - {gender} (B={B}, K={K}) ===")
    
    combination_idx = config.get("combination", 5)
    combination = get_feature_combination(combination_idx)
    target = 'TAC2'
    
    # File handling: read the one file for this gender directly (see GENDER_FILES).
    if gender not in GENDER_FILES:
        raise ValueError(f"Unknown gender {gender!r}; expected one of {list(GENDER_FILES)}")
    onlyfiles = [GENDER_FILES[gender]]
        
    # Device setup (Mac Metal vs CUDA vs CPU)
    if torch.cuda.is_available(): device = torch.device("cuda")
    elif torch.backends.mps.is_available(): device = torch.device("mps")
    else: device = torch.device("cpu")
    print(f" > Device: {device}")
    
    for f in onlyfiles:
        file_path = os.path.join(config["input_dir"], f)
        
        # Load and clean Data
        df_raw = pd.read_csv(file_path)
        df = df_raw.dropna(subset=combination + [target]).reset_index(drop=True)
        if len(df) < 10: continue

        # Prepare full dataset tensors
        data_all, data_0, data_1 = load_data_safe(df, combination, target, group_col)
        
        X_eval = data_all['x']
        X_eval_torch = torch.tensor(X_eval, dtype=torch.float32).to(device)
        Y_all = df[group_col].values 
        N_eval = len(df)
        
        # Track global indices to ensure we only test on patients NOT in training (OOB)
        idx_global_0 = np.where(Y_all == 0)[0]
        idx_global_1 = np.where(Y_all == 1)[0]
        
        X0, Y0, W0 = data_0['x'], data_0['y'], data_0['w']
        X1, Y1, W1 = data_1['x'], data_1['y'], data_1['w']
        N0, N1 = len(X0), len(X1)
        
        # Storage for OOB AUCs per patient
        patient_aucs = [[] for _ in range(N_eval)]
        
        print(f" > Dataset: {os.path.basename(file_path)} | N={N_eval}")
        
        # --- BOOTSTRAP LOOP ---
        for b in tqdm(range(B), desc="Bootstrap"):
            
            # Resample indices with replacement
            idx_sel_local_0 = np.random.choice(N0, size=N0, replace=True)
            idx_sel_local_1 = np.random.choice(N1, size=N1, replace=True)
            
            X0_b, Y0_b, W0_b = X0[idx_sel_local_0], Y0[idx_sel_local_0], W0[idx_sel_local_0]
            X1_b, Y1_b, W1_b = X1[idx_sel_local_1], Y1[idx_sel_local_1], W1[idx_sel_local_1]
            
            # Accumulators for Cross-Validation ensemble
            mu0_accum = np.zeros(N_eval); sig0_accum = np.zeros(N_eval)
            mu1_accum = np.zeros(N_eval); sig1_accum = np.zeros(N_eval)
            
            resid_std_0_pool = []
            resid_std_1_pool = []
            
            kf = KFold(n_splits=K)
            
            # --- Model 0 (Control Group) Training ---
            for tr_idx, val_idx in kf.split(X0_b):
                m0 = train_hetero_model(X0_b[tr_idx], Y0_b[tr_idx], W0_b[tr_idx], config, device)
                
                m0.eval()
                with torch.no_grad():
                    # Predict on EVERYONE (we will filter for OOB later)
                    m, s = m0(X_eval_torch)
                    mu0_accum += m.cpu().numpy() / K
                    sig0_accum += s.cpu().numpy() / K
                    
                    # Capture Residuals from the Fold's Validation set
                    x_val_fold = torch.tensor(X0_b[val_idx], dtype=torch.float32).to(device)
                    y_val_real = Y0_b[val_idx].flatten()
                    
                    m_val, s_val = m0(x_val_fold)
                    m_val = m_val.cpu().numpy()
                    s_val = s_val.cpu().numpy()
                    
                    # Standardized Residual = (True - Pred) / Sigma
                    eps = (y_val_real - m_val) / s_val
                    resid_std_0_pool.extend(eps)

            # --- Model 1 (Case Group) Training ---
            for tr_idx, val_idx in kf.split(X1_b):
                m1 = train_hetero_model(X1_b[tr_idx], Y1_b[tr_idx], W1_b[tr_idx], config, device)
                
                with torch.no_grad():
                    m, s = m1(X_eval_torch)
                    mu1_accum += m.cpu().numpy() / K
                    sig1_accum += s.cpu().numpy() / K
                    
                    # Capture Residuals
                    x_val_fold = torch.tensor(X1_b[val_idx], dtype=torch.float32).to(device)
                    y_val_real = Y1_b[val_idx].flatten()
                    m_val, s_val = m1(x_val_fold)
                    eps = (y_val_real.flatten() - m_val.cpu().numpy().flatten()) / s_val.cpu().numpy().flatten()
                    resid_std_1_pool.extend(eps)

            # Convert to numpy
            eps0_arr = np.array(resid_std_0_pool)
            eps1_arr = np.array(resid_std_1_pool)

            # Compute AUC using the Monte Carlo Helper
            auc_all = compute_empirical_auc(mu0_accum, mu1_accum, 
                                            sig0_accum, sig1_accum, 
                                            eps0_arr, eps1_arr, n_mc=500)
            
            # --- Anti-Leakage Check (OOB Filter) ---
            # Identify which patients were used in training this bootstrap
            used_global_0 = set(idx_global_0[idx_sel_local_0])
            used_global_1 = set(idx_global_1[idx_sel_local_1])
            
            for i in range(N_eval):
                is_safe = False
                # If patient is Control (0), check if they were used in Control Training
                if Y_all[i] == 0:
                    if i not in used_global_0: is_safe = True
                # If patient is Case (1), check if they were used in Case Training
                else:
                    if i not in used_global_1: is_safe = True
                
                # Only save prediction if the patient was Out-Of-Bag (not seen by model)
                if is_safe:
                    patient_aucs[i].append(auc_all[i])
        
        # --- AGGREGATE RESULTS ---
        auc_mean, auc_lower, auc_upper = [], [], []
        for i in range(N_eval):
            samples = np.array(patient_aucs[i])
            if len(samples) > 5: 
                auc_mean.append(np.mean(samples))
                auc_lower.append(np.percentile(samples, 2.5))  # 95% CI Lower
                auc_upper.append(np.percentile(samples, 97.5)) # 95% CI Upper
            else:
                auc_mean.append(np.nan)
                auc_lower.append(np.nan)
                auc_upper.append(np.nan)
                
        # Save to CSV
        df_res = df.copy()
        df_res['CROC_AUC_Mean'] = auc_mean
        df_res['CROC_CI_Lower'] = auc_lower
        df_res['CROC_CI_Upper'] = auc_upper
        
        out_dir = os.path.join(config["output_file"], group_col, os.path.splitext(f)[0], f"BootstrapOOB_Empirical_B{B}")
        os.makedirs(out_dir, exist_ok=True)
        df_res.to_csv(os.path.join(out_dir, "results_{}_{}.csv".format(group_col, gender)), index=False)
        
        # --- PLOTTING (Gaussian Smoothing) ---
        group_map = {
            "tres": "three",
            "cinco": "five",
            "ocho": "eight"
        }
        col_edad = 'RIDAGEYR.x' if 'RIDAGEYR.x' in df_res.columns else 'RIDAGEYR'
        if col_edad in df_res.columns and config.get("show_plots", True):
            
            # Group by Age to get raw means
            df_grouped = df_res.groupby(col_edad)[['CROC_AUC_Mean', 'CROC_CI_Lower', 'CROC_CI_Upper']].mean()
            
            # Apply Gaussian Rolling Window for professional smoothing
            window_size = 15
            sigma_val = 3.0
            df_smooth = df_grouped.rolling(window=window_size, win_type='gaussian', center=True, min_periods=3).mean(std=sigma_val)
            
            plt.figure(figsize=(10, 7))
            plt.fill_between(df_smooth.index, 
                             df_smooth['CROC_CI_Lower'], 
                             df_smooth['CROC_CI_Upper'], 
                             color='blue', alpha=0.15, label='95% CI (Smoothed)')
            plt.plot(df_smooth.index, df_smooth['CROC_AUC_Mean'], 
                     color='blue', linewidth=2.5, label='AUC Trend')
            
            plt.xlabel("Age (Years)")
            plt.ylabel("Estimated AUC")
            plt.title(f"Bootstrap OOB: {gender} {group_map.get(group_col, group_col)} survival years")
            plt.legend()
            plt.savefig(os.path.join(out_dir, "auc_plot_smooth_{}_{}.png".format(group_col, gender)), dpi=300)
            plt.show()
            plt.close()

In [ ]:
run_bootstrap_crossfitting('tres', 'female', config, B=100, K=5)

In [ ]:
run_bootstrap_crossfitting('tres', 'male', config, B=100, K=5)

In [ ]:
run_bootstrap_crossfitting('cinco', 'female', config, B=100, K=5)

In [ ]:
run_bootstrap_crossfitting('cinco', 'male', config, B=100, K=5)

In [ ]:
run_bootstrap_crossfitting('ocho', 'female', config, B=100, K=5)

In [ ]:
run_bootstrap_crossfitting('ocho', 'male', config, B=100, K=5)